# AIST-FYP Colab Evaluation Notebook

This notebook runs **RAGTruth** and **CiteEval** evaluation using the repository codebase.

## What this notebook does
1. Mounts Google Drive and clones the repo
2. Installs project dependencies
3. Hydrates prebuilt retrieval artifacts and benchmark data
4. Runs RAGTruth evaluation
5. Runs CiteEval evaluation (system and optional metric track)
6. Saves outputs back to Drive

## 📂 Artifact Placement Checklist

Before running the evaluation, ensure your repository has the necessary artifacts in the following structure:

- `data/`
  - `indexes/`
    - `{STRATEGY}/` (e.g., `production/`)
      - `faiss.index`
      - `metadata.pkl`
  - `processed/`
    - `wiki_chunks_{STRATEGY}.jsonl`
- `benchmark/`
  - `RAGTruth/`
    - `dataset/`
  - `CiteEval/`

If these are not in the repository you clone, you must upload them manually to the `/content/AIST-FYP/` directory after cloning.

## 🔑 Setup API Keys (Colab Secrets)

To run the evaluation, you likely need API keys for the LLM providers (DeepSeek, OpenAI, etc.). Colab provides a secure way to store these using the **Secrets** feature.

1. Click on the **Key icon** (Secrets) in the left sidebar.
2. Add a new secret with the name:
   - `OPENAI_API_KEY`: Your OpenAI API key.
   - `DEEPSEEK_API_KEY`: Your DeepSeek API key.
   - `HUGGINGFACE_TOKEN`: (Optional) For gated models.
3. Make sure to toggle the **"Notebook access"** switch to **ON** for this notebook.

The next cell will automatically load these secrets into the environment variables used by the project scripts.

In [ ]:
# ==============================
# Setup API Keys (Secrets)
# ==============================
try:
    from google.colab import userdata
    import os

    # Define keys to load
    secret_keys = ["OPENAI_API_KEY", "DEEPSEEK_API_KEY", "HUGGINGFACE_TOKEN"]
    loaded_any = False

    for key in secret_keys:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
                print(f"✅ Loaded secret: {key}")
                loaded_any = True
        except Exception:
            # Silently skip if not found or no access
            pass
    
    if not loaded_any:
        print("ℹ️ No secrets loaded. If you need API keys, add them via the 🔑 (Secrets) tab.")
except ImportError:
    print("⚠️ 'google.colab.userdata' not found. If running locally, please export your API keys manually.")

In [ ]:
# ==============================
# Parameters (edit this cell)
# ==============================
REPO_URL = "https://github.com/xiashuidaolaoshuren/AIST-FYP.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/AIST-FYP"
COLAB_ENV_PROJECT = "colab/env"
COLAB_UV_EXTRAS = ["evaluation"]

RUN_RAGTRUTH = True
RUN_CITEEVAL = True
RUN_CITEEVAL_METRIC = False  # requires metric data + human labels

# Evaluation knobs
RAGTRUTH_SPLIT = "test"
RAGTRUTH_MAX_SAMPLES = 10   # None for full split
RAGTRUTH_BATCH_SIZE = 10
RAGTRUTH_EVAL_MODE = "ragtruth_eval"  # ragtruth_eval | normal
STRATEGY = "production"      # development | validation | production

# CiteEval CLI knobs (direct script call)
CITEEVAL_PROVIDER = "deepseek"       # deepseek | openai
CITEEVAL_MODEL_NAME = ""             # empty => script default by provider
CITEEVAL_MAX_EXAMPLES = 10
CITEEVAL_METRIC_SPLIT = "test"       # dev | test
CITEEVAL_DRY_RUN_FIRST = True         # run --dry-run before actual execution

# ⚠️ REMINDER: Ensure the following artifacts are placed in the repo correctly:
# 1. FAISS Indexes: {REPO_DIR}/data/indexes/{STRATEGY}/faiss.index
# 2. Index Metadata: {REPO_DIR}/data/indexes/{STRATEGY}/metadata.pkl
# 3. Wikipedia Chunks: {REPO_DIR}/data/processed/wiki_chunks_{STRATEGY}.jsonl
# 4. Benchmarks: {REPO_DIR}/benchmark/RAGTruth/dataset/ and {REPO_DIR}/benchmark/CiteEval/
ARTIFACTS_IN_PROJECT = True

# Output location in your Drive
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/AIST-FYP-colab-outputs"

# Optional: existing full-pipeline output to convert for CiteEval system track
SYSTEM_INPUT_JSON = ""  # e.g., outputs/full_pipeline_queries_xxx.json

In [ ]:
import os
import re
import json
import shutil
import subprocess
from pathlib import Path

def run(cmd, cwd=None, check=True, stream=False):
    print(f"\n$ {cmd}")
    if stream:
        process = subprocess.Popen(
            cmd,
            shell=True,
            cwd=cwd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
        )
        out_lines = []
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            out_lines.append(line)
        process.wait()
        completed = subprocess.CompletedProcess(
            args=cmd,
            returncode=process.returncode,
            stdout="".join(out_lines),
            stderr=None,
        )
    else:
        completed = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
        if completed.stdout:
            print(completed.stdout)

    if completed.returncode != 0:
        if not stream and completed.stderr:
            print(completed.stderr)
        if check:
            raise RuntimeError(f"Command failed ({completed.returncode}): {cmd}")
    return completed

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def copytree_merge(src, dst):
    src_p = Path(src)
    dst_p = Path(dst)
    if not src_p.exists():
        return
    for p in src_p.rglob('*'):
        rel = p.relative_to(src_p)
        t = dst_p / rel
        if p.is_dir():
            t.mkdir(parents=True, exist_ok=True)
        else:
            t.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, t)

def exists_or_raise(path, msg):
    if not Path(path).exists():
        raise FileNotFoundError(f"{msg}: {path}")

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone repo
if Path(REPO_DIR).exists():
    print(f"Repo dir already exists: {REPO_DIR}")
else:
    run(f"git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")

run("git rev-parse --abbrev-ref HEAD", cwd=REPO_DIR)
run("git log -1 --oneline", cwd=REPO_DIR)

In [ ]:
# Install dependencies
run("python -m pip install -U pip wheel setuptools")
run("python -m pip install -U uv")

uv_project = Path(REPO_DIR) / COLAB_ENV_PROJECT
extras_args = " ".join(f"--extra {extra}" for extra in COLAB_UV_EXTRAS)
sync_cmd = f"uv sync --project {uv_project} {extras_args}"
result = run(sync_cmd, cwd=REPO_DIR, check=False)

if result.returncode == 0:
    uv_python = uv_project / ".venv" / "bin" / "python"
    os.environ["PATH"] = f"{uv_python.parent}:{os.environ.get('PATH', '')}"
    print(f"✅ uv sync complete: {uv_project}")
else:
    print('\n⚠️ uv sync failed. Falling back to pip requirements install...')
    requirements_path = Path(REPO_DIR) / 'requirements.txt'
    pytorch_index = 'https://download.pytorch.org/whl/cu121'
    install_cmd = f"pip install --extra-index-url {pytorch_index} -r {requirements_path}"
    fallback_result = run(install_cmd, cwd=REPO_DIR, check=False)

    if fallback_result.returncode != 0:
        print('\n⚠️ Full requirements install failed. Falling back to Colab-torch-compatible install...')
        filtered = []
        skip_prefixes = ('torch==', 'torchvision==', 'torchaudio==')
        for raw in requirements_path.read_text(encoding='utf-8').splitlines():
            line = raw.strip()
            if not line or line.startswith('#'):
                continue
            if any(line.startswith(prefix) for prefix in skip_prefixes):
                continue
            filtered.append(line)

        temp_req = Path(REPO_DIR) / 'requirements.colab.filtered.txt'
        temp_req.write_text('\n'.join(filtered) + '\n', encoding='utf-8')
        run(f"pip install -r {temp_req}", cwd=REPO_DIR)

# spaCy model required by verifier
run("python -m spacy download en_core_web_sm", cwd=REPO_DIR)

In [ ]:
# Runtime and API key setup
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Set keys in Colab before running CiteEval modules that need them:
# os.environ['DEEPSEEK_API_KEY'] = '...'
# os.environ['OPENAI_API_KEY'] = '...'

# Provider defaults from parameter cell
os.environ['CITEEVAL_PROVIDER'] = CITEEVAL_PROVIDER
os.environ['CITEEVAL_ROOT'] = str(Path(REPO_DIR) / 'benchmark/CiteEval')
extra_paths = [str(Path(REPO_DIR) / 'benchmark/CiteEval'), str(Path(REPO_DIR) / 'benchmark/CiteEval/src')]
existing_pp = os.environ.get('PYTHONPATH', '')
os.environ['PYTHONPATH'] = (existing_pp + os.pathsep if existing_pp else '') + os.pathsep.join(extra_paths)

print('CITEEVAL_PROVIDER =', os.environ.get('CITEEVAL_PROVIDER'))
print('CITEEVAL_MODEL_NAME =', CITEEVAL_MODEL_NAME if CITEEVAL_MODEL_NAME else '(script default)')
print('DEEPSEEK_API_KEY set =', bool(os.environ.get('DEEPSEEK_API_KEY')))
print('OPENAI_API_KEY set =', bool(os.environ.get('OPENAI_API_KEY')))

In [ ]:
# Artifacts check: using project-local folder structure (no external hydration)
if ARTIFACTS_IN_PROJECT:
    required_roots = [
        Path(REPO_DIR) / 'data',
        Path(REPO_DIR) / 'benchmark',
    ]
    for root in required_roots:
        if root.exists():
            print('Found:', root)
        else:
            print('Missing expected folder:', root)
else:
    print('ARTIFACTS_IN_PROJECT=False (no copy step configured).')

In [ ]:
# Validate required paths for selected strategy
repo = Path(REPO_DIR)
faiss_index = repo / f"data/indexes/{STRATEGY}/faiss.index"
index_meta = repo / f"data/indexes/{STRATEGY}/metadata.pkl"
chunks_file = repo / f"data/processed/wiki_chunks_{STRATEGY}.jsonl"
ragtruth_dataset = repo / 'benchmark/RAGTruth/dataset'

exists_or_raise(faiss_index, 'Missing FAISS index')
exists_or_raise(index_meta, 'Missing index metadata')
exists_or_raise(chunks_file, 'Missing processed chunks')

if RUN_RAGTRUTH:
    exists_or_raise(ragtruth_dataset, 'Missing RAGTruth dataset directory')

print('Preflight path checks passed.')

In [ ]:
# Create a Colab-specific config file based on config.yaml
import yaml

base_config = Path(REPO_DIR) / 'config.yaml'
colab_config = Path(REPO_DIR) / 'config.colab.yaml'

with open(base_config, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

# Keep GPU as requested; fail fast if unavailable
cfg['processing']['device'] = 'cuda'
cfg['verification']['nli']['device'] = 'cuda'
cfg['verification']['self_agreement']['device'] = 'cuda'

# Ensure evaluation mode defaults
cfg.setdefault('processing', {}).setdefault('query_split', {})['enabled'] = False
cfg.setdefault('evaluation', {}).setdefault('benchmarks', {}).setdefault('ragtruth', {})['ragtruth_eval_mode'] = 'ragtruth_eval'

with open(colab_config, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print('Wrote', colab_config)

In [ ]:
# Run RAGTruth evaluation (direct script call)
if RUN_RAGTRUTH:
    max_samples_arg = '' if RAGTRUTH_MAX_SAMPLES is None else f" --max-samples {RAGTRUTH_MAX_SAMPLES}"
    cmd = (
        f"python scripts/demo_ragtruth_eval.py --config config.colab.yaml --split {RAGTRUTH_SPLIT} "
        f"--batch-size {RAGTRUTH_BATCH_SIZE} --strategy {STRATEGY} --save-results "
        f"--ragtruth-eval-mode {RAGTRUTH_EVAL_MODE}{max_samples_arg}"
    )
    run(cmd, cwd=REPO_DIR)
else:
    print('RUN_RAGTRUTH=False, skipped.')

In [ ]:
# Summarize latest RAGTruth result
ragtruth_out = Path(REPO_DIR) / 'outputs/ragtruth_eval'
if ragtruth_out.exists():
    files = sorted(ragtruth_out.glob('*.json'), key=lambda p: p.stat().st_mtime, reverse=True)
    if files:
        latest = files[0]
        print('Latest RAGTruth file:', latest)
        payload = json.loads(latest.read_text(encoding='utf-8'))
        metrics = payload.get('metrics', {}).get('overall', payload.get('overall', {}))
        if metrics:
            print({k: metrics.get(k) for k in ['num_samples', 'accuracy', 'precision', 'recall', 'f1']})

        ensure_dir(DRIVE_OUTPUT_DIR)
        shutil.copy2(latest, Path(DRIVE_OUTPUT_DIR) / latest.name)
        print('Copied to Drive:', Path(DRIVE_OUTPUT_DIR) / latest.name)
    else:
        print('No RAGTruth JSON found.')
else:
    print('No outputs/ragtruth_eval directory found.')

In [ ]:
# Prepare CiteEval system input + preflight checks (direct script call)
if RUN_CITEEVAL:
    if SYSTEM_INPUT_JSON:
        cmd = (
            f"python scripts/convert_to_citeeval.py --input {SYSTEM_INPUT_JSON} "
            f"--output benchmark/CiteEval/data/system_eval/my_pipeline_results.json --strategy {STRATEGY}"
        )
        run(cmd, cwd=REPO_DIR)
    else:
        print('SYSTEM_INPUT_JSON is empty. Using an existing system_eval JSON if present.')

    provider = CITEEVAL_PROVIDER.strip().lower()
    if provider == 'deepseek' and not os.environ.get('DEEPSEEK_API_KEY'):
        raise EnvironmentError('DEEPSEEK_API_KEY is required when CITEEVAL_PROVIDER=deepseek')
    if provider == 'openai' and not os.environ.get('OPENAI_API_KEY'):
        raise EnvironmentError('OPENAI_API_KEY is required when CITEEVAL_PROVIDER=openai')

    system_eval_dir = Path(REPO_DIR) / 'benchmark/CiteEval/data/system_eval'
    system_jsons = sorted(system_eval_dir.glob('*.json')) if system_eval_dir.exists() else []
    if not system_jsons:
        raise FileNotFoundError('No system_eval JSON found for CiteEval system track.')
    print('System eval input candidates:', [p.name for p in system_jsons][:5])

    # Optional dry-run to validate command wiring before real run
    if CITEEVAL_DRY_RUN_FIRST:
        dry_run_cmd = 'python scripts/run_citebench_eval.py --track both --dry-run'
        run(dry_run_cmd, cwd=REPO_DIR)
else:
    print('RUN_CITEEVAL=False, skipped.')

In [ ]:
# Run CiteEval (direct script call; system track always, metric optional)
if RUN_CITEEVAL:
    system_eval_dir = Path(REPO_DIR) / 'benchmark/CiteEval/data/system_eval'
    system_jsons = sorted(system_eval_dir.glob('*.json'), key=lambda p: p.stat().st_mtime, reverse=True)
    system_input = system_jsons[0]

    provider_arg = f" --provider {CITEEVAL_PROVIDER}" if CITEEVAL_PROVIDER else ''
    model_arg = f" --model-name {CITEEVAL_MODEL_NAME}" if CITEEVAL_MODEL_NAME else ''
    max_examples_arg = '' if CITEEVAL_MAX_EXAMPLES is None else f" --max-examples {CITEEVAL_MAX_EXAMPLES}"

    system_cmd = (
        f"python scripts/run_citebench_eval.py --track system --system-input {system_input}"
        f"{provider_arg}{model_arg}{max_examples_arg}"
    )
    run(system_cmd, cwd=REPO_DIR)

    if RUN_CITEEVAL_METRIC:
        metric_cmd = (
            f"python scripts/run_citebench_eval.py --track metric --metric-split {CITEEVAL_METRIC_SPLIT}"
            f"{provider_arg}{model_arg}{max_examples_arg}"
        )
        run(metric_cmd, cwd=REPO_DIR)
else:
    print('RUN_CITEEVAL=False, skipped.')

In [ ]:
# Export evaluation outputs to Drive
ensure_dir(DRIVE_OUTPUT_DIR)
targets = [
    Path(REPO_DIR) / 'outputs',
    Path(REPO_DIR) / 'benchmark/CiteEval/data/system_eval_outputs',
    Path(REPO_DIR) / 'benchmark/CiteEval/data/metric_eval_outputs',
]

for t in targets:
    if t.exists():
        dest = Path(DRIVE_OUTPUT_DIR) / t.name
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(t, dest)
        print('Exported:', t, '->', dest)
    else:
        print('Skip missing:', t)

print('Done. Outputs available at:', DRIVE_OUTPUT_DIR)